## Import libraries, activate spark session

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import re
from tqdm import tqdm

import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, lit, udf
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
country_configs = {
    "ARG": {
        "name": "Argentina",
        "contact_keywords": ["contacto", "contáctenos", "observaciones"],
        "about_keywords": ["acerca de", "sobre nosotros", "quiénes somos"],
        "price_patterns": [r'ARS\s?\d+', r'\$\s?\d+', r'\d+\.\d{2}\s?\$'],
        "input_table": "DATABASE.SCHEMA.WEBSITE_SCRAPED_DATA_ARG",
        "output_table": "DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_ARG"
    },
    "BRA": {
        "name": "Brazil",
        "contact_keywords": ["contato", "fale conosco", "entre em contato"],
        "about_keywords": ["sobre", "quem somos", "informações"],
        "price_patterns": [r'R\$\s?\d+', r'\d+\.\d{2}\s?R\$'],
        "input_table": "DATABASE.SCHEMA.WEBSITE_SCRAPED_DATA_BRA",
        "output_table": "DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_BRA"
    },
    "COL": {
        "name": "Colombia",
        "contact_keywords": ["contacto", "contáctanos", "servicio al cliente"],
        "about_keywords": ["acerca de", "sobre nosotros", "nosotros"],
        "price_patterns": [r'COP\s?\d+', r'\$\s?\d+', r'\d+\.\d{2}\s?\$'],
        "input_table": "DATABASE.SCHEMA.WEBSITE_SCRAPED_DATA_COL",
        "output_table": "DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_COL"
    },
    "MEX": {
        "name": "Mexico",
        "contact_keywords": ["contacto", "contáctanos", "atención al cliente"],
        "about_keywords": ["acerca de", "sobre nosotros", "quienes somos"],
        "price_patterns": [r'MXN\s?\d+', r'\$\s?\d+', r'\d+\.\d{2}\s?\$'],
        "input_table": "DATABASE.SCHEMA.WEBSITE_SCRAPED_DATA_MEX",
        "output_table": "DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_MEX"
    },
    "CAN": {
        "name": "Canada",
        "contact_keywords": ["contact", "get in touch", "customer service"],
        "about_keywords": ["about us", "our story", "about"],
        "price_patterns": [r'CAD\s?\d+', r'\$\s?\d+', r'\d+\.\d{2}\s?\$'],
        "input_table": "DATABASE.SCHEMA.WEBSITE_SCRAPED_DATA_CAN",
        "output_table": "DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_CAN"
    },
    "JAM": {
        "name": "Jamaica",
        "contact_keywords": ["contact", "get in touch", "customer service"],
        "about_keywords": ["about us", "our story", "about"],
        "price_patterns": [r'J\$\s?\d+', r'\d+\.\d{2}\s?J\$'],
        "input_table": "DATABASE.SCHEMA.WEBSITE_SCRAPED_DATA_JAM",
        "output_table": "DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_JAM"
    }
}


In [ ]:
-- CREATE STAGE DATABASE.SCHEMA.STAGE_WEBSITES;

# Choose the country_code and run
* DEU, FIN, HUN, DNK, ISR, GRC

### Load Dataset and CONFIG

In [ ]:
country_code = "BRA"

In [ ]:
CONTACT_KEYWORDS = r"|".join(map(re.escape, country_configs[country_code]["contact_keywords"]))
ABOUT_KEYWORDS = r"|".join(map(re.escape, country_configs[country_code]["about_keywords"]))
PRICE_PATTERNS = country_configs[country_code]["price_patterns"]
INPUT_TABLE = country_configs[country_code]["input_table"]
OUTPUT_TABLE = country_configs[country_code]["output_table"]
print(country_code, '\n', CONTACT_KEYWORDS, '\n', ABOUT_KEYWORDS, '\n', PRICE_PATTERNS, '\n', INPUT_TABLE, '\n', OUTPUT_TABLE)

In [ ]:
df = (
    session.table(INPUT_TABLE)
    .select("WEBSITE", "HTML_CONTENT")
    .filter(col("status") == "Success")
)
df_count = df.count()
print('{:,}'.format(df_count), len(df.columns))

In [ ]:
df.limit(3).to_pandas()

### Function to extract HTML content

In [ ]:
# Set max HTML processing size (avoid memory spikes)
# MAX_HTML_SIZE = 500000

# UDF to extract features from HTML
@udf(name="extract_features_udf", 
     is_permanent=True,
     replace=True,
     stage_location="@STAGE_WEBSITES", 
     packages=["beautifulsoup4", "lxml"])
def extract_features(html_content: str, CONTACT_KEYWORDS: str, ABOUT_KEYWORDS: str, PRICE_PATTERNS: list) -> dict:
    """Extracts text, structure, and product-related features from HTML."""
    if not html_content:
        return {
            "word_count": 0, "title_length": 0, "has_contact_page": 0, 
            "has_about_page": 0, "num_links": 0, "num_images": 0, 
            "num_scripts": 0, "has_price_listings": 0
        }

    try:
        soup = BeautifulSoup(html_content, 'lxml')
        # soup = BeautifulSoup(html_content[:MAX_HTML_SIZE], 'lxml')

        # Text Features
        text = soup.get_text(" ", strip=True)
        word_count = len(text.split())

        title = soup.title.string.strip() if soup.title and soup.title.string else ""
        has_contact = bool(re.search(CONTACT_KEYWORDS, text, re.I))
        has_about = bool(re.search(ABOUT_KEYWORDS, text, re.I))

        # Structural Features
        num_links = len(soup.find_all("a"))
        num_images = len(soup.find_all("img"))
        num_scripts = len(soup.find_all("script"))

        # Product Listings Detection
        # price_patterns = [r'€\s?\d+', r'\d+\.\d{2}\s?€', r'\$\s?\d+', r'\d+\.\d{2}\s?\$']
        has_price = any(re.search(pattern, text, re.I) for pattern in PRICE_PATTERNS)

        return {
            "word_count": word_count, "title_length": len(title), "has_contact_page": int(has_contact),
            "has_about_page": int(has_about), "num_links": num_links, "num_images": num_images,
            "num_scripts": num_scripts, "has_price_listings": int(has_price)
        }

    except Exception:
        return {"word_count": 0, "title_length": 0, "has_contact_page": 0, 
                "has_about_page": 0, "num_links": 0, "num_images": 0, 
                "num_scripts": 0, "has_price_listings": 0}

### Process and generate output table

In [ ]:
# Apply UDF to Snowflake DataFrame
df_processed = df.with_column("features", extract_features(col("HTML_CONTENT")))

# Explode dictionary into separate columns
df_final = df_processed.select(
    col("WEBSITE"),
    col("features")["word_count"].alias("word_count"),
    col("features")["title_length"].alias("title_length"),
    col("features")["has_contact_page"].alias("has_contact_page"),
    col("features")["has_about_page"].alias("has_about_page"),
    col("features")["num_links"].alias("num_links"),
    col("features")["num_images"].alias("num_images"),
    col("features")["num_scripts"].alias("num_scripts"),
    col("features")["has_price_listings"].alias("has_price_listings")
)

# Compute quality score in Snowflake SQL
df_final = df_final.with_column(
    "quality_score",
    (
        (col("word_count") > 300).cast("int") * 2 +
        (col("word_count") > 100).cast("int") * 1 +
        (col("title_length") > 10).cast("int") * 1 +
        col("has_contact_page") * 1 +
        col("has_about_page") * 1 +
        (col("num_links") > 10).cast("int") * 1 +
        (col("num_images") > 5).cast("int") * 1 +
        col("has_price_listings") * 3
    )
)

# Save results back to Snowflake
df_final.write.mode("overwrite").save_as_table(OUTPUT_TABLE)
print(f"Table created: {OUTPUT_TABLE}")

* DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_ARG
* DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_BRA
* DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_COL
* DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_MEX
* DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_CAN
* DATABASE.SCHEMA.WEBSITE_QUALITY_SCORES_JAM